# Task 5 (Code), SDE Simulations
## `05-stochastic-code.ipynb`

Computational companion to the Task 5 theoretical notebook. Adds runnable simulations of
the stochastic Lorenz system using the Euler-Maruyama method, with visualizations of how
noise affects the attractor, the synchronization quality, and the bit error rate of
binary chaos shift keying.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f9f9f9',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
    'figure.dpi':       110,
})

# Project parameters from Tasks 1 and 2
SIGMA = 16.0
R     = 45.6
B     = 4.0

rng = np.random.default_rng(seed=2026)

def drift(state, b=B):
    u, v, w = state
    return np.array([
        SIGMA * (v - u),
        R * u - v - 20.0 * u * w,
        5.0 * u * v - b * w,
    ])

def euler_maruyama(state0, dt, n_steps, sigma_n, b=B):
    """Integrate the stochastic Lorenz system.
    sigma_n: noise intensity (0 means deterministic).
    """
    states = np.empty((n_steps + 1, 3))
    states[0] = state0
    sqrt_dt = np.sqrt(dt)
    for k in range(n_steps):
        f = drift(states[k], b=b)
        dW = rng.standard_normal(3) * sqrt_dt
        states[k+1] = states[k] + f * dt + sigma_n * dW
    return states

print('Setup complete. Euler-Maruyama solver ready.')

In [ ]:
# Compare deterministic vs stochastic trajectories
DT      = 0.001
T_TOT   = 30.0
N_STEPS = int(T_TOT / DT)
t       = np.linspace(0, T_TOT, N_STEPS + 1)
ic      = [0.1, 0.0, 0.0]

traj_det      = euler_maruyama(ic, DT, N_STEPS, sigma_n=0.0)
traj_low_n    = euler_maruyama(ic, DT, N_STEPS, sigma_n=0.05)
traj_high_n   = euler_maruyama(ic, DT, N_STEPS, sigma_n=0.30)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Stochastic Lorenz, u(t) at three noise levels',
             fontsize=13, fontweight='bold')

for ax, traj, label, color in zip(
        axes,
        [traj_det, traj_low_n, traj_high_n],
        ['Deterministic ($\\sigma_n=0$)', 'Low noise ($\\sigma_n=0.05$)', 'High noise ($\\sigma_n=0.30$)'],
        ['steelblue', '#27ae60', '#c0392b']):
    ax.plot(t, traj[:, 0], color=color, lw=0.6)
    ax.set_ylabel('u(t)')
    ax.set_title(label, fontsize=11)

axes[-1].set_xlabel('Time, s')
plt.tight_layout()
plt.show()

In [ ]:
# 3D attractor: deterministic vs noisy
fig = plt.figure(figsize=(13, 5))
fig.suptitle('Strange Attractor under Additive Noise',
             fontsize=13, fontweight='bold')

for i, (traj, label) in enumerate([
        (traj_det,    'Deterministic'),
        (traj_low_n,  '$\\sigma_n = 0.05$'),
        (traj_high_n, '$\\sigma_n = 0.30$')]):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], lw=0.3, color='steelblue', alpha=0.7)
    ax.set_xlabel('u'); ax.set_ylabel('v'); ax.set_zlabel('w')
    ax.set_title(label)
    ax.view_init(elev=22, azim=-55)

plt.tight_layout()
plt.show()
print('As noise increases, the fine structure of the attractor blurs and the trajectory broadens.')

In [ ]:
# Noise effect on synchronization
# Both transmitter and receiver evolve under independent noise;
# receiver is driven by transmitter's noisy u(t).
def coupled_em(ic_tx, ic_rx, dt, n_steps, sigma_n, b=B):
    state_tx = np.array(ic_tx, dtype=float)
    state_rx = np.array(ic_rx, dtype=float)
    out_tx = np.empty((n_steps + 1, 3))
    out_rx = np.empty((n_steps + 1, 3))
    out_tx[0] = state_tx
    out_rx[0] = state_rx
    sqrt_dt = np.sqrt(dt)
    for k in range(n_steps):
        f_tx = drift(state_tx, b=b)
        dW_tx = rng.standard_normal(3) * sqrt_dt
        state_tx = state_tx + f_tx * dt + sigma_n * dW_tx
        u_tx = state_tx[0]
        # Receiver: u_r autonomous, v_r and w_r driven by u_tx
        ur, vr, wr = state_rx
        f_rx = np.array([
            SIGMA * (vr - ur),
            R * u_tx - vr - 20.0 * u_tx * wr,
            5.0 * u_tx * vr - b * wr,
        ])
        dW_rx = rng.standard_normal(3) * sqrt_dt
        state_rx = state_rx + f_rx * dt + sigma_n * dW_rx
        out_tx[k+1] = state_tx
        out_rx[k+1] = state_rx
    return out_tx, out_rx

DT_S    = 0.001
T_S     = 20.0
N_S     = int(T_S / DT_S)
t_s     = np.linspace(0, T_S, N_S + 1)
ic_tx   = [0.1, 0.0, 0.0]
ic_rx   = [2.0, -1.5, 3.0]   # different from TX

noise_levels = [0.0, 0.05, 0.15, 0.30]
results = {}
for sn in noise_levels:
    rng = np.random.default_rng(seed=2026)   # reset for fair comparison
    tx, rx = coupled_em(ic_tx, ic_rx, DT_S, N_S, sigma_n=sn)
    err = tx[:, 0] - rx[:, 0]
    results[sn] = (tx, rx, err)

fig, axes = plt.subplots(len(noise_levels), 1, figsize=(12, 9), sharex=True)
fig.suptitle('Synchronization Error $e(t) = u(t) - u_r(t)$ at Increasing Noise',
             fontsize=13, fontweight='bold')

for ax, sn in zip(axes, noise_levels):
    err = results[sn][2]
    ax.plot(t_s, err, color='#c0392b', lw=0.6)
    rms = np.sqrt(np.mean(err[len(err)//3:]**2))
    ax.set_ylabel(f'$\\sigma_n = {sn}$\n\nrms = {rms:.3f}')
    ax.axhline(0, color='black', lw=0.6, linestyle=':')

axes[-1].set_xlabel('Time, s')
plt.tight_layout()
plt.show()

In [ ]:
# Synchronization rms error vs noise intensity
noise_sweep = np.linspace(0, 0.5, 11)
rms_errors  = []

for sn in noise_sweep:
    rng = np.random.default_rng(seed=2026)
    tx, rx = coupled_em(ic_tx, ic_rx, DT_S, N_S, sigma_n=sn)
    e = tx[:, 0] - rx[:, 0]
    rms_errors.append(np.sqrt(np.mean(e[N_S//3:]**2)))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(noise_sweep, rms_errors, 'o-', color='steelblue', lw=2, markersize=7)
ax.set_xlabel('Noise intensity $\\sigma_n$')
ax.set_ylabel('Synchronization error rms')
ax.set_title('Synchronization Quality Degrades Linearly with Noise',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

print('Approximate slope (rms error per unit noise):',
      f'{np.polyfit(noise_sweep, rms_errors, 1)[0]:.3f}')

In [ ]:
# Bit Error Rate (BER) for binary CSK under noise
B0_BIT = 4.0
B1_BIT = 4.4
BIT_DUR  = 5.0
DT_B     = 0.001
PTS_BIT  = int(BIT_DUR / DT_B)
N_TRIALS = 32   # bits per noise level

def csk_step(state_tx, state_rx, dt, n, sigma_n, b_tx, b_rx=B0_BIT):
    err_sum = 0.0
    sqrt_dt = np.sqrt(dt)
    for _ in range(n):
        f_tx = drift(state_tx, b=b_tx)
        state_tx = state_tx + f_tx * dt + sigma_n * rng.standard_normal(3) * sqrt_dt
        u_tx = state_tx[0]
        ur, vr, wr = state_rx
        f_rx = np.array([
            SIGMA * (vr - ur),
            R * u_tx - vr - 20.0 * u_tx * wr,
            5.0 * u_tx * vr - b_rx * wr,
        ])
        state_rx = state_rx + f_rx * dt + sigma_n * rng.standard_normal(3) * sqrt_dt
        err_sum += (u_tx - state_rx[0]) ** 2
    return state_tx, state_rx, err_sum / n

def measure_csk_ber(sigma_n):
    bits = rng.integers(0, 2, size=N_TRIALS).tolist()
    state_tx = np.array([0.1, 0.0, 0.0])
    state_rx = np.array([0.5, -0.3, 0.2])
    powers = []
    for b in bits:
        b_val = B1_BIT if b == 1 else B0_BIT
        state_tx, state_rx, p = csk_step(state_tx, state_rx, DT_B, PTS_BIT, sigma_n, b_tx=b_val)
        powers.append(p)
    powers = np.array(powers)
    bits_a = np.array(bits)
    p0_mean = powers[bits_a == 0].mean() if (bits_a == 0).any() else 0
    p1_mean = powers[bits_a == 1].mean() if (bits_a == 1).any() else 1
    theta = (p0_mean + p1_mean) / 2.0
    detected = (powers > theta).astype(int)
    ber = float(np.mean(detected != bits_a))
    return ber

noise_ber = np.linspace(0.0, 0.4, 9)
ber_curve = []
for sn in noise_ber:
    rng = np.random.default_rng(seed=int(sn*1000) + 1)
    ber_curve.append(measure_csk_ber(sn))
    print(f'  sigma_n = {sn:.3f}  ->  BER = {ber_curve[-1]:.3f}')

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(noise_ber, ber_curve, 'o-', color='#c0392b', lw=2, markersize=7)
ax.axhline(0.5, color='black', linestyle='--', lw=1, label='Random guessing (BER 0.5)')
ax.set_xlabel('Noise intensity $\\sigma_n$')
ax.set_ylabel('Bit error rate')
ax.set_title('CSK Bit Error Rate vs Noise',
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

## Summary of the noise simulations

| Quantity | Behavior under noise |
|----------|---------------------|
| Attractor structure | Fine structure blurs as $\sigma_n$ increases. The two-lobed butterfly remains visible up to roughly $\sigma_n \approx 0.3$. |
| Synchronization error rms | Grows roughly linearly with $\sigma_n$. Quality is acceptable for $\sigma_n < 0.1$. |
| CSK bit error rate | Rises slowly at first, then degrades sharply once noise exceeds the parameter mismatch in the b-shift. The system becomes unusable as BER approaches 0.5. |

These results are consistent with the theoretical bounds in the Task 5 theory notebook: the
noise floor of the recovered message scales linearly with $\sigma_n$, and detection reliability
depends on the contrast between the matched and mismatched error powers.

For the physical circuit in Task 4, this gives a concrete noise-tolerance specification:
the analog noise floor of the implementation should stay below $\sigma_n \approx 0.05$ to
preserve high-quality message recovery and low bit error rate.